# 前端集成概述与集成库

来源:
- https://docs.langchain.com/mcp/frontend/overview
- https://docs.langchain.com/mcp/frontend/integrations/overview
- https://docs.langchain.com/mcp/frontend/ai-elements
- https://docs.langchain.com/mcp/frontend/assistant-ui
- https://docs.langchain.com/mcp/frontend/copilotkit
- https://docs.langchain.com/mcp/frontend/openui

本笔记覆盖LangGraph前端集成的整体架构、useStream Hook以及四大前端框架(I Elements、assistant-ui、CopilotKit、OpenUI)。

---
## 1. 前端集成概述

LangGraph前端集成架构的核心是`useStream` Hook，它封装了与LangGraph Server的通信逻辑。

### 架构总览

```
[React/Next.js 前端]                    [LangGraph Server]
    │                                           │
    ├── useStream() Hook ──────────────→ POST /runs/stream
    │       │                                   │
    │       ├── 管理连接状态                      ├── 执行Agent图
    │       ├── 处理SSE事件流                     ├── 返回流式事件
    │       ├── 维护消息历史                      └── 管理线程状态
    │       └── 处理中断/审批
    │
    ├── UI组件库 (AI Elements / assistant-ui / CopilotKit)
    └── 自定义UI组件
```

### useStream Hook 核心功能

| 功能 | 说明 |
|------|------|
| 流式消息 | 实时接收Agent的流式输出 |
| 线程管理 | 自动创建和管理对话线程 |
| 中断处理 | 检测并处理Human-in-the-Loop中断 |
| 状态同步 | 保持客户端状态与服务端同步 |
| 历史记录 | 加载和切换历史对话 |

---
## 2. useStream 使用示例

```tsx
// TypeScript React
import { useStream } from "@langchain/langgraph-sdk/react";

function ChatInterface() {
  const {
    threadId,
    messages,
    submit,
    isLoading,
    error,
    interrupts
  } = useStream({
    apiUrl: "http://localhost:8123",
    assistantId: "customer_support",
  });

  const [input, setInput] = useState("");

  const handleSubmit = async () => {
    if (!input.trim()) return;
    await submit({ messages: [{ role: "human", content: input }] });
    setInput("");
  };

  // 处理中断 (审批请求等)
  React.useEffect(() => {
    if (interrupts.length > 0) {
      const lastInterrupt = interrupts[interrupts.length - 1];
      console.log("审批请求:", lastInterrupt);
    }
  }, [interrupts]);

  if (error) return <div>错误: {error.message}</div>;

  return (
    <div className="chat-container">
      <div className="messages">
        {messages.map((msg, i) => (
          <MessageBubble key={i} message={msg} />
        ))}
        {isLoading && <TypingIndicator />}
      </div>
      <div className="input-area">
        <input
          value={input}
          onChange={(e) => setInput(e.target.value)}
          disabled={isLoading}
          placeholder="输入消息..."
        />
        <button onClick={handleSubmit} disabled={isLoading}>
          发送
        </button>
      </div>
    </div>
  );
}
```

---
## 3. AI Elements

AI Elements 是LangChain官方的React组件库，提供开箱即用的聊天UI组件。

### 安装

```bash
npm install @langchain/ai-elements
```

### 核心组件

| 组件 | 说明 |
|------|------|
| `<ThreadList>` | 对话线程列表 |
| `<ChatWindow>` | 聊天窗口容器 |
| `<MessageList>` | 消息列表渲染 |
| `<MessageInput>` | 消息输入框 |
| `<TypingIndicator>` | 键入指示器 |
| `<ToolCallResult>` | 工具调用结果展示 |

### 基本使用

```tsx
// TypeScript React
import {
  ChatWindow,
  MessageList,
  MessageInput,
  ThreadList
} from "@langchain/ai-elements";
import { useStream } from "@langchain/langgraph-sdk/react";

function App() {
  const stream = useStream({
    apiUrl: "http://localhost:8123",
    assistantId: "my_agent",
  });

  return (
    <div className="app-layout">
      <ThreadList {...stream} />
      <ChatWindow {...stream}>
        <MessageList {...stream} />
        <MessageInput {...stream} />
      </ChatWindow>
    </div>
  );
}
```

---
## 4. assistant-ui

assistant-ui 是一个功能完整的开源聊天UI框架，深度集成LangGraph。

### 安装

```bash
npm install assistant-ui @assistant-ui/react @assistant-ui/langgraph
```

### 核心特性

- 流式消息渲染
- 工具调用可视化
- 线程管理
- 消息编辑和重试
- Markdown + 代码高亮
- 自定义组件插槽

### 使用示例

```tsx
// TypeScript React
import { AssistantRuntimeProvider } from "@assistant-ui/react";
import { useLangGraphRuntime } from "@assistant-ui/langgraph";
import { Thread } from "@assistant-ui/react";

function MyApp() {
  const runtime = useLangGraphRuntime({
    apiUrl: "http://localhost:8123",
    assistantId: "my_agent",
  });

  return (
    <AssistantRuntimeProvider runtime={runtime}>
      <Thread />
    </AssistantRuntimeProvider>
  );
}

// 自定义样式
// 支持CSS变量覆盖:
// --assistant-ui-primary-color
// --assistant-ui-font-family
// --assistant-ui-border-radius
```

---
## 5. CopilotKit

CopilotKit 是一个将AI助手嵌入任何React应用的框架，支持LangGraph后端。

### 安装

```bash
npm install @copilotkit/react-core @copilotkit/react-ui
```

### 核心概念

| 概念 | 说明 |
|------|------|
| **CopilotKit** | 根Provider组件 |
| **CopilotSidebar** | 侧边栏聊天UI |
| **CopilotPopup** | 浮动聊天按钮 |
| **useCopilotAction** | 定义前端可执行动作 |
| **useCopilotReadable** | 向前端共享状态 |

### 使用示例

```tsx
// TypeScript React
import { CopilotKit } from "@copilotkit/react-core";
import { CopilotSidebar } from "@copilotkit/react-ui";
import "@copilotkit/react-ui/styles.css";

function App() {
  return (
    <CopilotKit
      runtimeUrl="/api/langgraph"  // 代理到LangGraph Server
      agent="customer_support"
    >
      <div className="app-content">
        <h1>我的应用</h1>
        <CopilotSidebar
          defaultOpen={true}
          labels={{
            title: "AI助手",
            initial: "你好！有什么可以帮助你的？",
          }}
        />
      </div>
    </CopilotKit>
  );
}

// 定义前端动作
function MyComponent() {
  useCopilotAction({
    name: "showChart",
    description: "显示数据图表",
    parameters: [{ name: "data", type: "object" }],
    render: ({ args }) => <ChartComponent data={args.data} />,
  });
  return <div>...</div>;
}
```

---
## 6. OpenUI

OpenUI 是一个快速原型工具，可以通过对话生成和编辑UI组件。

### 特点

- 自然语言生成UI
- 实时预览和迭代
- 导出为React/HTML代码
- 支持自定义组件库

### 使用方式

```bash
# 启动OpenUI
docker run -p 3000:3000 ghcr.io/wandb/openui

# 访问 http://localhost:3000
# 描述你需要的UI组件，AI生成代码
```

```tsx
// TypeScript - OpenUI 生成的示例组件
function GeneratedChatCard({ title, messages, onSend }) {
  return (
    <div className="max-w-md mx-auto bg-white rounded-xl shadow-md">
      <div className="p-4 border-b">
        <h2 className="text-lg font-semibold">{title}</h2>
      </div>
      <div className="p-4 h-96 overflow-y-auto">
        {messages.map((msg, i) => (
          <div key={i} className={`mb-2 p-2 rounded ${
            msg.role === 'user' ? 'bg-blue-100 ml-8' : 'bg-gray-100 mr-8'
          }`}>
            {msg.content}
          </div>
        ))}
      </div>
      <div className="p-4 border-t flex gap-2">
        <input className="flex-1 p-2 border rounded" />
        <button onClick={onSend} className="px-4 py-2 bg-blue-500 text-white rounded">
          发送
        </button>
      </div>
    </div>
  );
}
```

---
## 7. 框架对比

| 特性 | AI Elements | assistant-ui | CopilotKit | OpenUI |
|------|-------------|--------------|------------|--------|
| 官方维护 | ✅ LangChain | ✅ 社区 | ✅ 商业 | ✅ Weights & Biases |
| 开箱即用 | 基础组件 | 完整UI | 完整UI+状态同步 | 原型生成 |
| 自定义程度 | 高 (组件级) | 中 (主题+插槽) | 中 (主题配置) | 极高 (生成代码) |
| LangGraph原生 | ✅ | ✅ | ✅ | 需配置 |
| 流式支持 | ✅ | ✅ | ✅ | N/A |
| 工具调用UI | ✅ | ✅ | ✅ | 需自定义 |
| HITL支持 | ✅ | ✅ | ✅ | 需自定义 |
| 适用场景 | 深度定制 | 快速集成 | 嵌入现有应用 | 原型设计 |

## 8. 参考链接

- [前端集成概述](https://docs.langchain.com/mcp/frontend/overview)
- [集成库概述](https://docs.langchain.com/mcp/frontend/integrations/overview)
- [AI Elements](https://docs.langchain.com/mcp/frontend/ai-elements)
- [assistant-ui](https://docs.langchain.com/mcp/frontend/assistant-ui)
- [CopilotKit](https://docs.langchain.com/mcp/frontend/copilotkit)
- [OpenUI](https://docs.langchain.com/mcp/frontend/openui)
- [LangGraph SDK React](https://www.npmjs.com/package/@langchain/langgraph-sdk)
- [assistant-ui 文档](https://www.assistant-ui.com/)
- [CopilotKit 文档](https://docs.copilotkit.ai/)